# Stage 05 — Diagnostics

Run automated checks and write `data/results/diagnostics_flags.json`.
Follow `skills/stage_05.md` for detailed instructions.

In [ ]:
from paths import *
import json
import numpy as np

spec      = json.loads(PAPER_SPEC.read_text())
rep_check = json.loads(REPLICATION_CHECK.read_text())
rep_res   = json.loads(REPLICATION_RESULTS.read_text())
dml_res   = json.loads(DML_RESULTS.read_text())

# Published main coefficient (first in main_results)
pub_coef = spec['main_results'][0]['coef'] if spec.get('main_results') else None
dml_coef = dml_res['preferred_coef']
dml_lo   = dml_res['preferred_ci_lo']
dml_hi   = dml_res['preferred_ci_hi']

print(f'Published main coef: {pub_coef}')
print(f'DML preferred coef: {dml_coef}  CI: [{dml_lo}, {dml_hi}]')

In [ ]:
def flag(value, pass_cond, warn_cond, note):
    if pass_cond:
        return {'status': 'PASS', 'value': value, 'note': note}
    elif warn_cond:
        return {'status': 'WARN', 'value': value, 'note': note}
    else:
        return {'status': 'FAIL', 'value': value, 'note': note}

checks = {}

# Check 1: Replication pass
worst = rep_check['worst_rel_diff_pct']
checks['replication_pass'] = flag(
    worst,
    pass_cond=(worst < 5),
    warn_cond=(worst < 15),
    note=f'{worst:.1f}% max deviation'
)

# Check 2: DML direction
if pub_coef is not None and dml_coef is not None:
    same_sign = (pub_coef * dml_coef) > 0
    rel_diff = abs(dml_coef - pub_coef) / abs(pub_coef) if pub_coef != 0 else float('nan')
    checks['dml_direction'] = flag(
        dml_coef,
        pass_cond=(same_sign and rel_diff < 0.5),
        warn_cond=(same_sign),
        note='same sign' if same_sign else 'SIGN CHANGE'
    )
else:
    checks['dml_direction'] = {'status': 'WARN', 'value': None, 'note': 'could not compare'}

# Check 3: CI coverage
if pub_coef is not None and dml_lo is not None:
    inside = dml_lo <= pub_coef <= dml_hi
    checks['dml_ci_coverage'] = flag(
        pub_coef,
        pass_cond=inside,
        warn_cond=True,
        note='published coef inside DML CI' if inside else 'published coef outside DML CI'
    )
else:
    checks['dml_ci_coverage'] = {'status': 'WARN', 'value': None, 'note': 'could not check'}

# Check 4: Nuisance quality
nuisance = dml_res.get('nuisance_scores', {})
r2_out = nuisance.get('r2_outcome', 0)
r2_trt = nuisance.get('r2_treatment', 0)
checks['nuisance_quality'] = flag(
    {'r2_outcome': r2_out, 'r2_treatment': r2_trt},
    pass_cond=(r2_out > 0.3 and r2_trt > 0.3),
    warn_cond=(r2_out > 0.1 and r2_trt > 0.1),
    note=f'r2_outcome={r2_out:.2f}, r2_treatment={r2_trt:.2f}'
)

# Check 5: Sample size
n = dml_res['n_obs']
checks['sample_size'] = flag(
    n,
    pass_cond=(n >= 50),
    warn_cond=(n >= 30),
    note=f'n={n}'
)

# Summarise
statuses = [v['status'] for v in checks.values()]
overall = 'FAIL' if 'FAIL' in statuses else ('WARN' if 'WARN' in statuses else 'PASS')
blocking = [k for k, v in checks.items() if v['status'] == 'FAIL']
warnings = [f"{k}: {v['note']}" for k, v in checks.items() if v['status'] == 'WARN']

diagnostics = {
    'overall': overall,
    'checks': checks,
    'blocking_issues': blocking,
    'warnings': warnings
}

DIAGNOSTICS_FLAGS.write_text(json.dumps(diagnostics, indent=2))
print(f'\nOverall: {overall}')
print(f'✓ {DIAGNOSTICS_FLAGS}')
for k, v in checks.items():
    icon = {'PASS': '✓', 'WARN': '⚠', 'FAIL': '✗'}[v['status']]
    print(f'  {icon} {k}: {v["note"]}')